# 🧪 SWIN PEFT UNIVARIATE HYPERPARAMETER TUNING SUITE (11 TRỤC ĐỘC LẬP)
### Khảo Sát Đơn Biến Toàn Diện (One-Factor-At-A-Time) Cho Swin Transformer với NCM Backbone Evaluation

* **Ngày cập nhật:** 2026-10-22
* **Khắc phục triệt để:**
  1. Tự động giải mã cấu trúc thư mục lồng nhau sâu trên Kaggle (`car_data/car_data/train`), chấm dứt lỗi `num_samples=0`.
  2. Đồng bộ chuẩn luồng thực thi `SwinTransformer` (khắc phục `pos_drop` và thứ tự `downsample` trước `blocks`).
* **Changelog / Modifications (v1.6 - Bộ Đệm Đĩa Tự Động Cho Cả ImageNet-R & CIFAR-100):**
  - **Phân luồng thông minh (Auto-Threshold $\ge 10,000$ mẫu):** Tự động nhận diện quy mô dataset:
    + Dataset lớn ($\ge 10,000$ mẫu: `cifar100` 50k ảnh $\to 7.5\text{ GB}$ đĩa, `imagenet_r` 24k ảnh $\to 3.6\text{ GB}$ đĩa): Tự động kích hoạt `Uint8MmapEvalDataset` trên `/tmp` (NVMe SSD), triệt tiêu hoàn toàn cạm bẫy nhân đôi bộ nhớ của `torch.stack` gây sập trần $30\text{ GiB}$ CPU RAM của Kaggle.
    + Dataset vừa & nhỏ ($< 10,000$ mẫu: `cars196` 8.1k ảnh, `cub200` 6.0k ảnh): Giữ nguyên nạp trực tiếp vào RAM (Zero-Disk I/O) cho tốc độ tuyệt đối.
  - **Đồng bộ hóa Normalization:** Tự động điều hướng mean/std theo `aug_type` (ImageNet cho `imagenet_r`/`cars`/`cub`, CIFAR cho `cifar100`).
* **Changelog / Modifications (v1.5 - Nâng Cấp Bộ Đệm Đĩa Siêu Tốc 7.5GB Uint8 Mmap Trên `/tmp`):**
  - **Khắc phục nghẽn CPU & Chống OOM RAM cho CIFAR-100:** Thay vì resize $32 \times 32 \to 224 \times 224$ lặp đi lặp lại trên CPU qua từng trục thử nghiệm (tốn $\sim 33$ phút) hoặc nạp $30.1\text{ GB}$ float32 gây sập RAM $30\text{ GB}$ của Kaggle, triển khai cơ chế **`Uint8MmapEvalDataset`** trên thư mục `/tmp` (ổ cứng NVMe SSD).
  - **Tối ưu dung lượng & thời gian:** Lưu toàn bộ $50,000$ ảnh dưới dạng mảng $224 \times 224 \times 3$ `uint8` chỉ chiếm đúng $\mathbf{7.5\text{ GB}}$ đĩa (trên tổng số $55.6\text{ GB}$ trống của Kaggle). Thời gian tạo bộ đệm 1 lần duy nhất chỉ $\sim 20\text{s}$.
  - **Tăng tốc trích xuất đặc trưng $\sim 7,000\text{ ảnh/s}$:** Ánh xạ bộ nhớ ảo (virtual memory page cache) giúp nạp dữ liệu tức thì cho cả `val_loader` (10k ảnh) và `all_tr_loader` (40k ảnh), giảm thời gian đánh giá NCM xuống còn $\sim 5\text{s}$, tiêu thụ RAM thực tế $< 1\text{ GB}$.
* **Changelog / Modifications (v1.4 - Khắc Phục Lỗi Nạp CIFAR-100 & Chống Tràn 30GB RAM):**
  - **Sửa Lỗi IndexError 16388 (Cell 4 & Cell 8):** Khắc phục triệt để lỗi len(raw_train) == 2 do cấu trúc tuple (data_arr, targets_arr) của CIFAR-100. Tính toán chính xác số mẫu 50,000 thay vì chỉ nạp 2 mẫu đầu tiên.
  - **Tối ưu RAM Chống OOM (Cell 4 & Cell 8):** Bổ sung RemappedSubset đọc trực tiếp từ mảng 153.6 MB uint8 trong RAM thay vì cố gắng stack 50,000 tensor float32 224x224 (tiêu tốn 30.1 GB gây sập Kaggle RAM). Chỉ nạp 10,000 tensor Validation vào RAM (~5.7 GB) giúp đánh giá siêu tốc 1.5s/lần.
  - **Tích Hợp seed_cache:** Chuẩn bị al_loader và 	1_ds 1 lần duy nhất cho mỗi seed, tái sử dụng tức thời cho toàn bộ 11 trục tuning và 5 tổ hợp Joint.
* **Changelog / Modifications (v1.3 - Khắc Phục Triệt Để LayerNorm & Bổ Sung Joint Verification):**
  - **Sửa Lỗi Khóa Trọng Số `swin.norm` (Cell 5):** Loại bỏ hoàn toàn hai lớp `self.ln3` và `self.ln4` (`elementwise_affine=True`) chưa được huấn luyện. Trích xuất trực tiếp `s4_tokens = self.swin.norm(x)` để giữ trọn vẹn trọng số $\gamma, \beta$ pre-trained trên ImageNet-22K của Microsoft; với Stage 3 áp dụng `F.layer_norm` không affine và chuẩn hóa năng lượng RMS: `f3 * (f4_rms / f3_rms)`, đồng bộ 100% với kiến trúc chuẩn `exp_swin_so_kac_ablation_3.ipynb`, giải phóng `eff_rank` lên 500+ và phục hồi không gian đặc trưng.
  - **Bổ Sung Giai Đoạn 2 - Kiểm Chứng Tổ Hợp Kết Hợp (Joint Combination Verification, Cell 8):** Giải quyết giới hạn của phương pháp One-Factor-At-A-Time (tránh cạm bẫy tích tụ bước nhảy khi gộp các biến tăng tốc riêng rẽ). Tự động chạy và xếp hạng 5 tổ hợp siêu tham số đại diện (`0_Anchor_Default`, `1_Joint_BS32_LR2e3`, `2_Joint_BS32_LR5e3`, `3_Joint_BS64_LR1e2`, `4_Raw_JSON_Best`) trên cả 3 seeds để tìm ra Champion Hyperparameter Set chung cuộc.
  - **Hỗ Trợ `EXECUTION_MODE` Linh Hoạt (Cell 3):** Cho phép người dùng chuyển đổi tức thì giữa `"FULL_SWEEP_AND_JOINT"` (~8 phút), `"JOINT_ONLY"` (~2-3 phút), và `"UNIVARIATE_ONLY"` (~6 phút).
* **Changelog / Modifications (v1.2 - Siêu Tối Ưu Tốc Độ RAM & GPU):**
  - **In-Memory RAM Pre-caching (Zero-Disk I/O):** Tận dụng 30 GB RAM của Kaggle, nạp toàn bộ 632 ảnh Task 1 vào RAM (`CachedTrainSubset`) và tiền tính toán 8,144 ảnh đánh giá thành tensor bộ nhớ (`precompute_eval_tensors`, ~1.2 GB RAM). Xóa bỏ hoàn toàn nghẽn cổ chai đĩa I/O (loại bỏ hơn 800,000 lần đọc file từ đĩa).
  - **Tối ưu hóa GPU Memory & Batch Size:** Đẩy batch size trích xuất feature từ 128 lên 256 với `num_workers=0` (in-memory DMA streaming), kích hoạt `pin_memory = True`. Tốc độ quét mỗi tham số tăng gấp 10 lần (chỉ còn ~1.5s mỗi lần đánh giá NCM).
  - **Khuyến nghị cấu hình Kaggle Accelerator:** Hướng dẫn tối ưu sang GPU P100 hoặc GPU T4 x1 để tránh lãng phí 2x GPU quota của chế độ Dual GPU (T4 x2).
* **Changelog / Modifications (v1.1):**
  - **Ghi Gradient Từng Epoch (Epoch-by-Epoch Gradient Tracking):** In và ghi nhận toàn bộ `LR`, `Loss`, `p_grad`, `c_grad` tại mọi epoch đơn lẻ ($1, 2, \dots, E$) với độ chính xác 6 chữ số thập phân (`.6f`), giúp quan sát chi tiết động thái hội tụ và trạng thái gradient của adapter/backbone.
  - **Chuẩn hóa AMP Scaler:** Gọi `scaler.unscale_(opt)` trước khi đo `grad_norm()` để đảm bảo gradient phản ánh đúng độ lớn FP32 thực tế (không bị phóng đại bởi dynamic loss scale).
  - **Lưu trữ Trajectory Toàn Diện:** Mỗi cấu hình thử nghiệm lưu mảng `epoch_history` chứa toàn bộ diễn biến qua từng epoch vào JSON kết quả để phục vụ vẽ đồ thị và phân tích sâu.
### 🛡️ Chiến Lược Phân Chia Dữ Liệu: 3 Seed Hoán Vị Lớp & Val Rời Rạc 0% Overlap
Kế thừa chuẩn mực từ `Fixed_Split_Export_4Datasets.ipynb` (Phương án A):
1. **Hoán vị thứ tự lớp (Class Order Permutation):** Mỗi seed ($0, 1, 2$) sử dụng `rng_task = np.random.RandomState(seed)` để hoán vị thứ tự 196 lớp, đảm bảo Task 1 ở mỗi seed học các nhóm lớp hoàn toàn khác nhau.
2. **Tập Val rời rạc tuyệt đối (0% Overlap):** Chia Stratified 5-Fold trên từng lớp với `base_seed=1993` cố định:
   - **Seed 0:** Dùng **Fold 0** làm tập Val ($pprox 1,629$ ảnh).
   - **Seed 1:** Dùng **Fold 1** làm tập Val ($pprox 1,629$ ảnh, **0% trùng lặp** với Fold 0).
   - **Seed 2:** Dùng **Fold 2** làm tập Val ($pprox 1,629$ ảnh, **0% trùng lặp** với Fold 0 & 1).
3. **Độ phủ dữ liệu vượt trội:** Khi chạy trên cả 3 seed, tổng số ảnh validation độc lập được đánh giá là **$1,629 	imes 3 = 5,009$ ảnh (chiếm 61.5% toàn bộ dữ liệu train)** mà **hoàn toàn không đụng vào 1 ảnh nào của tập Test thật** (Bảo toàn 100% Zero Test Leakage).

### ⚖️ Chiến Lược Tối Ưu Hóa: Tại Sao Anchor Baseline Dùng Adam Cố Định LR (Constant Learning Rate)?
Trong thực nghiệm Continual Prompt Tuning (APT, ICCV 2025) và các notebook đối chứng của dự án:
1. **Chuẩn đối chứng công bằng (Apples-to-Apples):** ViT-APT sử dụng thuần túy `torch.optim.Adam` với LR không đổi (`prompt_lr = const`, `betas=(0.9, 0.999)`) trong suốt các epoch của Task 1.
2. **Tránh bẫy đóng băng sớm (Premature Freezing):** Khi bật Cosine Annealing, LR ở các epoch cuối bị triệt tiêu tới 17 lần ($0.0005 	o 0.00003$), khiến các tham số PEFT bị đình trệ cập nhật, làm sai lệch đánh giá về số epochs cần thiết.
3. **Cấu hình Mốc Chuẩn (Anchor Baseline):** Đặt mặc định `use_cosine_sched = False`, `warmup_epochs = 0`, `weight_decay = 0.0`, `head_lr_ratio = 1.0` để phản ánh đúng 100% chuẩn Adam cố định gốc.
4. **Khảo sát mở rộng tại Trục 10:** Việc so sánh giữa **Adam Cố Định LR** (`False`) và **Cosine Annealing** (`True`) được dành riêng cho Trục 10 nhằm xác định một cách khoa học xem liệu biến thiên LR có cải thiện hình học không gian đặc trưng (NCM Acc) của Swin Transformer hay không.

* **Phương pháp khoa học:** **Univariate Sensitivity Analysis (Khảo sát độ nhạy đơn biến)**
  - Thiết lập **Anchor Baseline Config** chuẩn cố định cho tất cả các tham số.
  - Trên mỗi trục, chỉ **thay đổi duy nhất một siêu tham số** trong khi giữ nguyên 100% các tham số khác.
* **11 Trục Siêu Tham Số Toàn Diện:**
  1. `peft_lr` (Tốc độ học PEFT: `[0.0005, 0.001, 0.002, 0.005, 0.010, 0.020]`)
  2. `batch_size` (Kích thước Batch: `[32, 64, 128, 256]`)
  3. `epochs` (Số Epochs Task 1: `[10, 15, 20, 30, 40, 50]`)
  4. `warmup_epochs` (Epochs Warmup tuyến tính: `[0, 2, 5]`)
  5. `scale` ($s$ của Adapter/ConvPass: `[0.05, 0.10, 0.20, 0.50, 1.00]`)
  6. `rank` ($r$ chiều ẩn: `[8, 16, 32, 64]`)
  7. `adapter_dropout` (Tỷ lệ Dropout của nhánh PEFT: `[0.0, 0.1, 0.2]`)
  8. `head_lr_ratio` (Tỷ lệ $\frac{\text{head\_lr}}{\text{peft\_lr}}$: `[1.0, 2.0, 5.0]`)
  9. `weight_decay` (L2 penalty trong AdamW: `[0.0, 1e-5, 1e-4, 1e-3, 1e-2]`)
  10. `scheduler` (`AdamW + CosineAnnealingLR` vs `Adam Constant LR`)
  11. `ema_coeff` (Hệ số neo giữ trọng số ban đầu: `[0.3, 0.5, 0.7, 0.85, 1.0]`)
* **Đánh giá hình học biểu diễn (Representation Geometry):**
  - **Task 1 Classifier:** Dùng `CosineHead` ($d_{\text{in}} \to C_1$) chuẩn hóa cosine.
  - **Metric cốt lõi:** **NCM (Nearest Class Mean)** trên representation $\hat{\mu}_c = \frac{\mu_c}{\|\mu_c\|}$ để đo lường thuần túy chất lượng không gian đặc trưng của backbone mà không bị nhiễu bởi bộ phân loại.
  - **Diagnostic Metrics:** Đo đồng thời `NCM_Acc_All`, `NCM_Acc_T1`, `NCM_Acc_Unseen`, `FDR_Unseen`, `eff_rank`, `p_grad`, `c_grad`.
  - **Tối ưu tốc độ:** Chạy trên **Seed 0 (Val 80/20 Disjoint)**, tổng thời gian chỉ $\approx 10 \sim 13$ phút trên Kaggle GPU!

In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, THƯ VIỆN & THIẾT BỊ GPU
# ==============================================================================
import os
import sys
import math
import time
import json
import shutil
import pickle
import tempfile
import random
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "timm>=0.9.12", "-q"])
    import timm

IS_KAGGLE = os.path.exists("/kaggle")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
NUM_WORKERS = 4 if IS_KAGGLE else 2

print(f"[*] Environment: {'Kaggle GPU' if IS_KAGGLE else 'Local Environment'}")
print(f"[*] Device: {DEVICE} (AMP: {USE_AMP})")
if DEVICE == "cuda":
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")

def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(0)


In [ ]:
# ==============================================================================
# CELL 3: CẤU HÌNH ANCHOR MỐC CHUẨN & DANH MỤC 11 TRỤC TUNING ĐỘC LẬP
# ==============================================================================
# ==============================================================================
# BỘ ĐIỀU KHIỂN CHÍNH (MASTER USER CONTROLS)
# ==============================================================================
# 1. CHỌN DATASET CẦN TUNING: 'cars196', 'cub200', 'cifar100', 'imagenet_r'
ACTIVE_DATASET = "cars196"

# 2. CHỌN SEED THỰC NGHIỆM:
#    - [0]        : Quét siêu tốc trên Seed 0 (Val Fold 0: 1,707 ảnh, ~2 phút)
#    - [0, 1, 2]  : Quét trung bình 3 Seed chuẩn Fixed_Split_Export_4Datasets (Val Fold 0, 1, 2 rời rạc 0% overlap: 5,009 ảnh, ~6 phút)
ACTIVE_SEEDS = [0, 1, 2]

# 3. CHỌN DANH SÁCH TRỤC CẦN KHẢO SÁT: "ALL_AXES" hoặc danh sách ví dụ ["peft_lr", "epochs"]
ACTIVE_TUNING_AXES = "ALL_AXES"

# 4. CHỌN CHẾ ĐỘ THỰC THI (EXECUTION MODE):
#    - "FULL_SWEEP_AND_JOINT": Quét 11 trục đơn biến + Kiểm chứng 5 tổ hợp kết hợp (~8 phút)
#    - "JOINT_ONLY"          : Bỏ qua 11 trục đơn biến, chạy ngay 5 tổ hợp kết hợp (~2-3 phút)
#    - "UNIVARIATE_ONLY"     : Chỉ chạy 11 trục đơn biến (~6 phút)
EXECUTION_MODE = "FULL_SWEEP_AND_JOINT" 

DATASET_SPECS = OrderedDict([
    ("cars196", dict(
        dataset_name="Cars196", n_classes=196, task_sizes=[20]*9 + [16],
        aug_type="cars",
        kaggle_candidates=[
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data",
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder",
            "/kaggle/input/stanford-car-dataset-by-classes-folder/car_data/car_data",
            "/kaggle/input/stanford-car-dataset-by-classes-folder/car_data",
            "/kaggle/input/stanford-car-dataset-by-classes-folder",
            "/kaggle/input/stanford-cars-dataset/cars_train/cars_train",
            "/kaggle/input/stanford-cars-dataset",
            "/kaggle/input/cars196/cars_train",
            "/kaggle/input/cars196",
            "/kaggle/input/stanford-cars/cars_train",
            "./data/cars196", "./data/cars"
        ]
    )),
    ("cub200", dict(
        dataset_name="Cub200", n_classes=200, task_sizes=[20]*10,
        aug_type="cub",
        kaggle_candidates=[
            "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
            "/kaggle/input/cub-200-2011-by-classes-folder",
            "/kaggle/input/cyizhuo-cub-200-2011-by-classes-folder",
            "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder/train",
            "/kaggle/input/cub-200-2011-by-classes-folder/train",
            "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder/CUB_200_2011/images",
            "/kaggle/input/cub-200-2011-by-classes-folder/CUB_200_2011/images",
            "/kaggle/input/cub2002011",
            "/kaggle/input/cub-200-2011",
            "/kaggle/input/cub200",
            "./data/cub-200-2011-by-classes-folder",
            "./data/CUB_200_2011/images",
            "./data/CUB_200_2011",
            "./data/cub200"
        ]
    )),
    ("cifar100", dict(
        dataset_name="Cifar100", n_classes=100, task_sizes=[10]*10,
        aug_type="cifar",
        kaggle_candidates=[
            "/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
            "/kaggle/input/datasets/thohintrnhngc/cifar-100-python",
            "/kaggle/input/cifar-100-python/cifar-100-python",
            "/kaggle/input/cifar-100-python",
            "/kaggle/input/cifar100-python",
            "/kaggle/input/cifar100",
            "./data/cifar-100-python",
            "./data/cifar100"
        ]
    )),
    ("imagenet_r", dict(
        dataset_name="ImageNet_R", n_classes=200, task_sizes=[20]*10,
        aug_type="inr",
        kaggle_candidates=[
            "/kaggle/input/datasets/thohintrnhngc/imagenet-r/imagenet-r",
            "/kaggle/input/datasets/thohintrnhngc/imagenet-r",
            "/kaggle/input/imagenet-r/imagenet-r",
            "/kaggle/input/imagenet-r",
            "./data/imagenet-r",
            "./data/ImageNet_R"
        ]
    )),
])


# ==============================================================================
# 2. CẤU HÌNH MỐC CHUẨN CỐ ĐỊNH (ANCHOR BASELINE CONFIG)
DEFAULT_ANCHOR_CONFIG = {
    "backbone": "swin_small",
    "fusion": "F3_multistage",
    "peft": "P5_hybrid_conv_rpb",
    "peft_lr": 0.002,
    "head_lr_ratio": 1.0,       # Chuẩn APT gốc: head_lr = peft_lr
    "batch_size": 128,
    "epochs": 30,
    "warmup_epochs": 0,         # Adam Cố Định LR từ Epoch 1 (chuẩn APT gốc)
    "scale": 0.2,
    "rank": 16,
    "adapter_dropout": 0.0,     # Tỷ lệ dropout của Adapter
    "weight_decay": 0.0,        # Adam thuần túy không weight decay (chuẩn APT)
    "use_cosine_sched": False,  # Cố định LR 100% (không suy giảm cosine)
    "ema_coeff": 0.7,
}

# 3. DANH MỤC 11 TRỤC TUNING ĐỘC LẬP (UNIVARIATE TUNING AXES)
TUNING_AXES = OrderedDict([
    # TRỤC 1: Tốc độ học (Learning Rate) của PEFT
    ("peft_lr", {
        "param_name": "peft_lr",
        "values": [0.0005, 0.001, 0.002, 0.005, 0.010, 0.020],
        "desc": "Khảo sát Tốc độ học PEFT (Learning Rate) khi cố định các tham số khác"
    }),
    # TRỤC 2: Batch Size khi huấn luyện Task 1
    ("batch_size", {
        "param_name": "batch_size",
        "values": [32, 64, 128, 256],
        "desc": "Khảo sát Kích thước Batch huấn luyện Task 1"
    }),
    # TRỤC 3: Số Epochs huấn luyện Task 1 (Khảo sát điểm bão hòa / overfit)
    ("epochs", {
        "param_name": "epochs",
        "values": [10, 15, 20, 30, 40, 50],
        "desc": "Khảo sát Số Epochs huấn luyện Task 1 (chống overfit/underfit)"
    }),
    # TRỤC 4: Linear Warmup Epochs (0 = Cố định từ đầu chuẩn APT, 2 & 5 = Thử nghiệm Warmup)
    ("warmup_epochs", {
        "param_name": "warmup_epochs",
        "values": [0, 2, 5],
        "desc": "Khảo sát Warmup tuyến tính (0 = Cố định từ đầu chuẩn APT)"
    }),
    # TRỤC 5: Adapter / ConvPass Scale (s)
    ("scale", {
        "param_name": "scale",
        "values": [0.05, 0.10, 0.20, 0.50, 1.00],
        "desc": "Khảo sát Hệ số tỷ lệ Scale (s) của nhánh PEFT trên 20 blocks Swin"
    }),
    # TRỤC 6: Bottleneck Rank (r)
    ("rank", {
        "param_name": "rank",
        "values": [8, 16, 32, 64],
        "desc": "Khảo sát Chiều ẩn Bottleneck Rank (r) của Adapter / ConvPass"
    }),
    # TRỤC 7: Adapter Dropout (Chống quá khớp số mẫu ít)
    ("adapter_dropout", {
        "param_name": "adapter_dropout",
        "values": [0.0, 0.1, 0.2],
        "desc": "Khảo sát Tỷ lệ Dropout bên trong nhánh PEFT"
    }),
    # TRỤC 8: Tỷ lệ Head LR / PEFT LR
    ("head_lr_ratio", {
        "param_name": "head_lr_ratio",
        "values": [1.0, 2.0, 5.0],
        "desc": "Khảo sát Tỷ lệ tốc độ học của Classifier Head so với PEFT"
    }),
    # TRỤC 9: Trọng số phạt L2 (Weight Decay) trong AdamW
    ("weight_decay", {
        "param_name": "weight_decay",
        "values": [0.0, 1e-5, 1e-4, 1e-3, 1e-2],
        "desc": "Khảo sát Weight Decay của AdamW"
    }),
    # TRỤC 10: Bộ lập lịch LR (Adam Cố Định LR vs AdamW + CosineAnnealingLR)
    ("scheduler", {
        "param_name": "use_cosine_sched",
        "values": [False, True],
        "desc": "So sánh Adam Cố Định LR (Chuẩn APT) vs AdamW + CosineAnnealingLR"
    }),
    # TRỤC 11: Quét tức thời hệ số EMA Anchoring (0s extra training)
    ("ema_coeff", {
        "param_name": "ema_coeff",
        "values": [0.3, 0.5, 0.7, 0.85, 1.0],
        "desc": "Quét hệ số neo giữ trọng số ban đầu (Fast 0s Sweep)"
    }),
])

MASTER_OUT_DIR = "/kaggle/working/outputs/swin_tuning" if IS_KAGGLE else "./results/apt/swin_tuning"
os.makedirs(MASTER_OUT_DIR, exist_ok=True)
print(f"[*] Dataset: {ACTIVE_DATASET} | Active Seeds: {ACTIVE_SEEDS} ({len(ACTIVE_SEEDS)} seeds)")
print(f"[*] Anchor Baseline Config: {DEFAULT_ANCHOR_CONFIG}")
print(f"[*] Tổng số trục khảo sát: {len(TUNING_AXES)}")


In [ ]:
# ==============================================================================
# CELL 4: NẠP DỮ LIỆU, TỰ ĐỘNG GIẢI MÃ CẤU TRÚC LỒNG NHAU & CHIA 80/20 VAL DISJOINT
# ==============================================================================
class CIFAR100PickleDataset(Dataset):
    def __init__(self, root, train=True):
        split_file = "train" if train else "test"
        p_path = os.path.join(root, split_file)
        with open(p_path, "rb") as f:
            entry = pickle.load(f, encoding="latin1")
        self.data = entry["data"].reshape(-1, 3, 32, 32).transpose((0, 2, 3, 1))
        self.targets = entry["fine_labels"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return Image.fromarray(self.data[idx]), int(self.targets[idx])

class CachedTrainSubset(Dataset):
    def __init__(self, raw_samples, indices, remap_dict, transform, is_cifar=False):
        self.transform = transform
        self.samples = []
        for idx in indices:
            if is_cifar:
                data_arr, targets_arr = raw_samples
                img = Image.fromarray(data_arr[idx])
                y = remap_dict[int(targets_arr[idx])]
            else:
                p, y = raw_samples[idx]
                img = Image.open(p).convert("RGB")
                img.load()  # Tự động giải nén sẵn vào RAM, triệt tiêu đọc đĩa
                y = remap_dict[y]
            self.samples.append((img, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, y = self.samples[idx]
        return self.transform(img), y

class RemappedSubset(Dataset):
    """
    Dataset bọc dữ liệu cho cả dạng danh sách file ảnh [(path, label)] lẫn dạng mảng nén RAM (CIFAR-100).
    Hoàn toàn không tốn RAM khổng lồ nhưng vẫn đạt tốc độ cực đại nhờ đọc trực tiếp từ mảng numpy trong bộ nhớ.
    """
    def __init__(self, raw_samples, indices, remap_dict, transform, is_cifar=False):
        self.transform = transform
        self.is_cifar = is_cifar
        self.indices = list(indices)
        self.remap_dict = remap_dict
        if is_cifar:
            self.data_arr, self.targets_arr = raw_samples
        else:
            self.samples = raw_samples
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        raw_idx = self.indices[i]
        if self.is_cifar:
            img = Image.fromarray(self.data_arr[raw_idx])
            y = self.remap_dict[int(self.targets_arr[raw_idx])]
        else:
            p, y_orig = self.samples[raw_idx]
            img = Image.open(p).convert("RGB")
            y = self.remap_dict[y_orig]
        return self.transform(img), y

class Uint8MmapEvalDataset(Dataset):
    """
    Dataset tối ưu đọc trực tiếp từ file memory-mapped np.memmap (uint8) trên đĩa NVMe (/tmp).
    Chỉ nạp đúng batch đang xét vào RAM, triệt tiêu 100% nguy cơ OOM 30 GB RAM trên Kaggle.
    Chuyển đổi uint8 sang float32 và Normalize siêu tốc theo chuẩn PyTorch (~7,000 ảnh/giây).
    """
    def __init__(self, mmap_path, indices, remap_dict, targets, shape=(50000, 224, 224, 3), aug_type="cifar"):
        self.mmap_path = mmap_path
        self.shape = shape
        self.indices = list(indices)
        self.remap_dict = remap_dict
        self.targets = targets
        self.mmap_arr = None
        if aug_type == "cifar":
            mean = [0.5071, 0.4867, 0.4408]
            std = [0.2675, 0.2565, 0.2761]
        else:
            mean = [0.485, 0.456, 0.406]
            std = [0.229, 0.224, 0.225]
        self.normalize = T.Normalize(mean=mean, std=std)

    def _get_mmap(self):
        if self.mmap_arr is None:
            self.mmap_arr = np.memmap(self.mmap_path, dtype=np.uint8, mode='r', shape=self.shape)
        return self.mmap_arr

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        raw_idx = self.indices[i]
        mmap_arr = self._get_mmap()
        img_arr = mmap_arr[raw_idx]
        tensor = torch.from_numpy(img_arr.copy()).permute(2, 0, 1).float().div_(255.0)
        tensor = self.normalize(tensor)
        y = self.remap_dict[int(self.targets[raw_idx])]
        return tensor, y

class PreloadedTensorDataset(Dataset):
    def __init__(self, tensors, labels):
        self.tensors = tensors
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.tensors[idx], self.labels[idx]

def precompute_eval_tensors(raw_samples, indices, remap_dict, eval_tf, is_cifar=False, desc="Eval"):
    print(f"[*] Đang tiền xử lý & nạp {len(indices)} ảnh {desc} vào RAM (Zero-Disk I/O)...")
    t0 = time.time()
    tensors, labels = [], []
    for idx in indices:
        if is_cifar:
            data_arr, targets_arr = raw_samples
            img = Image.fromarray(data_arr[idx])
            y = remap_dict[int(targets_arr[idx])]
        else:
            p, y = raw_samples[idx]
            img = Image.open(p).convert("RGB")
            y = remap_dict[y]
        tensors.append(eval_tf(img))
        labels.append(y)
    t_stack = torch.stack(tensors, dim=0)
    l_stack = torch.tensor(labels, dtype=torch.long)
    ram_mb = (t_stack.element_size() * t_stack.nelement()) / (1024 * 1024)
    print(f"    -> Hoàn tất nạp {t_stack.shape[0]} tensor vào RAM trong {time.time() - t0:.1f}s (RAM sử dụng: {ram_mb:.1f} MB)")
    return PreloadedTensorDataset(t_stack, l_stack)

def build_transforms(aug_type):
    crop_size = 224
    if aug_type in ["cars", "cub", "inr"]:
        train_tf = T.Compose([
            T.Resize((256, 256)),
            T.RandomResizedCrop(crop_size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        train_tf = T.Compose([
            T.RandomResizedCrop(crop_size, scale=(0.08, 1.0)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
        ])
    eval_tf = T.Compose([
        T.Resize((256, 256)),
        T.CenterCrop(crop_size),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) if aug_type != "cifar"
        else T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
    ])
    return train_tf, eval_tf

def locate_dataset_root(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Không tìm thấy thư mục dataset. Đã thử: {candidates}")

def resolve_dataset_dirs(root):
    # 1. Nếu đường dẫn truyền vào trực tiếp trỏ vào thư mục con 'train' hoặc 'test', tự động lùi về thư mục cha
    norm_path = os.path.normpath(root)
    if os.path.basename(norm_path).lower() in ["train", "test"]:
        parent = os.path.dirname(norm_path)
        p_tr = os.path.join(parent, "train")
        p_te = os.path.join(parent, "test")
        if os.path.isdir(p_tr) and os.path.isdir(p_te):
            return p_tr, p_te

    # 2. Kiểm tra trực tiếp tại root
    tr = os.path.join(root, "train")
    te = os.path.join(root, "test")
    if os.path.isdir(tr) and os.path.isdir(te):
        return tr, te

    # 3. Kiểm tra các thư mục con phổ biến
    common_subpaths = [
        ("car_data", "car_data"),
        ("car_data",),
        ("cars_train",),
        ("CUB_200_2011", "images"),
        ("images",),
    ]
    for sub in common_subpaths:
        cand_root = os.path.join(root, *sub)
        tr = os.path.join(cand_root, "train")
        te = os.path.join(cand_root, "test")
        if os.path.isdir(tr) and os.path.isdir(te):
            return tr, te
        if os.path.isdir(cand_root):
            subdirs = [d for d in os.listdir(cand_root) if os.path.isdir(os.path.join(cand_root, d))]
            if len(subdirs) >= 10 and "train" not in subdirs:
                return cand_root, None

    # 4. Duyệt đệ quy tìm cặp train/test
    for dirpath, dirnames, _ in os.walk(root):
        if "train" in dirnames and "test" in dirnames:
            tr = os.path.join(dirpath, "train")
            te = os.path.join(dirpath, "test")
            subdirs = [d for d in os.listdir(tr) if os.path.isdir(os.path.join(tr, d))]
            if len(subdirs) >= 10:
                return tr, te

    # 5. Thư mục đơn lẻ chứa trực tiếp các lớp
    subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    if len(subdirs) >= 10:
        return root, None

    return None, None

def load_raw_dataset_samples(d_key, spec):
    root = locate_dataset_root(spec["kaggle_candidates"])
    train_samples, test_samples = [], []
    actual_root = root
    if d_key in ["cub200", "cars196", "imagenet_r"]:
        tr_dir, te_dir = resolve_dataset_dirs(root)
        if tr_dir is not None and te_dir is not None:
            actual_root = os.path.dirname(tr_dir)
            tr_classes = sorted([d for d in os.listdir(tr_dir) if os.path.isdir(os.path.join(tr_dir, d))])
            te_classes = sorted([d for d in os.listdir(te_dir) if os.path.isdir(os.path.join(te_dir, d))])
            all_classes = sorted(list(set(tr_classes) | set(te_classes)))
            c2i = {c: i for i, c in enumerate(all_classes)}
            for sub_dir, target in [(tr_dir, train_samples), (te_dir, test_samples)]:
                for c in all_classes:
                    cd = os.path.join(sub_dir, c)
                    if os.path.isdir(cd):
                        for fname in sorted(os.listdir(cd)):
                            if fname.lower().endswith((".jpg", ".png", ".jpeg")):
                                target.append((os.path.join(cd, fname), c2i[c]))
        elif tr_dir is not None:
            actual_root = tr_dir
            classes = sorted([d for d in os.listdir(tr_dir) if os.path.isdir(os.path.join(tr_dir, d))])
            c2i = {c: i for i, c in enumerate(classes)}
            for c in classes:
                cd = os.path.join(tr_dir, c)
                fnames = sorted([f for f in os.listdir(cd) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
                s_list = [(os.path.join(cd, fn), c2i[c]) for fn in fnames]
                n_tr = int(0.8 * len(s_list))
                train_samples.extend(s_list[:n_tr])
                test_samples.extend(s_list[n_tr:])
        else:
            raise ValueError(f"Không thể định vị thư mục train/test hoặc các lớp trong: {root}")
    elif d_key == "cifar100":
        def unpickle(file):
            with open(file, 'rb') as fo:
                return pickle.load(fo, encoding='latin1')
        tr_b = unpickle(os.path.join(root, "train"))
        te_b = unpickle(os.path.join(root, "test"))
        train_samples = (tr_b["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1), tr_b["fine_labels"])
        test_samples  = (te_b["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1), te_b["fine_labels"])

    n_tr = len(train_samples[1]) if d_key == "cifar100" else len(train_samples)
    n_te = len(test_samples[1]) if d_key == "cifar100" else len(test_samples)
    print(f"[*] Đã nạp dataset {d_key}: {n_tr} mẫu train, {n_te} mẫu test từ {actual_root}")
    if n_tr == 0:
        raise ValueError(f"LỖI: 0 mẫu ảnh được nạp cho {d_key} từ đường dẫn {root}!")
    return train_samples, test_samples

def setup_cil_seed(raw_train, seed, num_classes, task_sizes=None, n_splits=5, base_seed=1993, is_cifar=False):
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    # (a) Đảo thứ tự class vào các Task theo seed (Phương án A chuẩn Fixed_Split_Export_4Datasets)
    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    remap = {orig: new for new, orig in enumerate(class_order)}

    if is_cifar:
        _, targets = raw_train
        targets = np.asarray(targets)
    else:
        targets = np.asarray([y for _, y in raw_train])

    # (b) Chia Stratified Disjoint 5-Fold bên trong từng class -> lấy Fold[seed] làm Val (20%, 0% overlap)
    rng_split = np.random.RandomState(base_seed)
    tr_indices, val_indices = [], []
    for c in range(num_classes):
        c_idx = np.where(targets == c)[0]
        perm = rng_split.permutation(c_idx)
        folds = np.array_split(perm, n_splits)
        val_indices.extend(folds[seed].tolist())
        for k in range(n_splits):
            if k != seed:
                tr_indices.extend(folds[k].tolist())

    return class_order, remap, tr_indices, val_indices


In [ ]:
# ==============================================================================
# CELL 5: KIẾN TRÚC SWIN, PEFT (HỖ TRỢ DROPOUT) & TRAIN (HỖ TRỢ WARMUP + HEAD LR RATIO)
# ==============================================================================
SWIN_ARCH_MAP = {
    "swin_tiny":  "swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",
    "swin_small": "swin_small_patch4_window7_224.ms_in22k_ft_in1k",
}

class ConvPass2DAdapter(nn.Module):
    def __init__(self, dim, rank=16, dropout=0.0):
        super().__init__()
        self.down = nn.Linear(dim, rank, bias=False)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.conv = nn.Conv2d(rank, rank, kernel_size=3, padding=1, groups=rank, bias=False)
        self.up = nn.Linear(rank, dim, bias=False)
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.conv.weight, a=math.sqrt(5))
        nn.init.zeros_(self.up.weight)

    def forward(self, x):
        z = self.drop(self.act(self.down(x)))
        if z.ndim == 4:
            z_2d = z.permute(0, 3, 1, 2).contiguous()
            conv_out = self.conv(z_2d).permute(0, 2, 3, 1).contiguous()
        elif z.ndim == 3:
            B, L, C = z.shape
            side = int(math.isqrt(L))
            if side * side == L:
                z_2d = z.view(B, side, side, C).permute(0, 3, 1, 2).contiguous()
                conv_out = self.conv(z_2d).permute(0, 2, 3, 1).contiguous().view(B, L, C)
            else:
                conv_out = z
        else:
            conv_out = z
        return self.up(self.drop(self.act(conv_out)))

class SwinPEFTModule(nn.Module):
    def __init__(self, stage_dims, stage_depths, stage_heads, peft_mode="P5_hybrid_conv_rpb",
                 fusion_mode="F3_multistage", window_size=7, rank=16, scale=0.2, dropout=0.0, swin_model=None):
        super().__init__()
        self.peft_mode = peft_mode
        self.fusion_mode = fusion_mode
        self.scale = scale
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        if peft_mode == "P2_adapter":
            self.down_list, self.up_list = nn.ModuleList(), nn.ModuleList()
            for s_idx, d in enumerate(stage_depths):
                dim = stage_dims[s_idx]
                for _ in range(d):
                    down = nn.Linear(dim, rank, bias=False)
                    up   = nn.Linear(rank, dim, bias=False)
                    nn.init.kaiming_uniform_(down.weight, a=math.sqrt(5))
                    nn.init.zeros_(up.weight)
                    self.down_list.append(down)
                    self.up_list.append(up)
        elif peft_mode == "P4_convpass":
            self.convpass_list = nn.ModuleList()
            for s_idx, d in enumerate(stage_depths):
                dim = stage_dims[s_idx]
                for _ in range(d):
                    self.convpass_list.append(ConvPass2DAdapter(dim=dim, rank=rank, dropout=dropout))
        elif peft_mode == "P5_hybrid_conv_rpb":
            self.convpass_list = nn.ModuleList()
            self.delta_rpb = nn.ParameterList()
            for s_idx, d in enumerate(stage_depths):
                dim = stage_dims[s_idx]
                layer_idx = 2 + s_idx
                for b_idx in range(d):
                    self.convpass_list.append(ConvPass2DAdapter(dim=dim, rank=rank, dropout=dropout))
                    if swin_model is not None:
                        orig_bias = swin_model.layers[layer_idx].blocks[b_idx].attn.relative_position_bias_table
                        delta = nn.Parameter(torch.zeros_like(orig_bias))
                    else:
                        nh = stage_heads[s_idx]
                        num_rel_pos = (2 * window_size - 1) * (2 * window_size - 1)
                        delta = nn.Parameter(torch.zeros(num_rel_pos, nh))
                    self.delta_rpb.append(delta)

        self.init_state = {k: v.clone().detach() for k, v in self.state_dict().items()}
        self.raw_trained_state = None

    def save_trained_state(self):
        self.raw_trained_state = {k: v.clone().detach() for k, v in self.state_dict().items()}

    def apply_ema_anchoring(self, alpha=0.7):
        if self.raw_trained_state is None:
            self.save_trained_state()
        with torch.no_grad():
            for name, param in self.named_parameters():
                if name in self.raw_trained_state and name in self.init_state:
                    w_trained = self.raw_trained_state[name].to(device=param.device, dtype=param.dtype)
                    w_init    = self.init_state[name].to(device=param.device, dtype=param.dtype)
                    param.copy_(alpha * w_trained + (1.0 - alpha) * w_init)

    def grad_norm(self):
        tot = 0.0
        for p in self.parameters():
            if p.grad is not None:
                tot += p.grad.detach().data.norm(2).item() ** 2
        return math.sqrt(tot)

class SwinBackboneWrapper(nn.Module):
    def __init__(self, timm_name="swin_small_patch4_window7_224.ms_in22k_ft_in1k",
                 fusion_mode="F3_multistage", peft_mode="P5_hybrid_conv_rpb",
                 window_size=7, rank=16, scale=0.2, dropout=0.0):
        super().__init__()
        self.swin = timm.create_model(timm_name, pretrained=True, num_classes=0)
        self.fusion_mode = fusion_mode
        self.peft_mode = peft_mode
        self.dropout = dropout
        self.s3_dim = self.swin.layers[2].blocks[0].dim
        self.s4_dim = self.swin.layers[3].blocks[0].dim
        self.stage_dims = [self.s3_dim, self.s4_dim]
        self.stage_depths = [len(self.swin.layers[2].blocks), len(self.swin.layers[3].blocks)]
        self.stage_heads = [self.swin.layers[2].blocks[0].attn.num_heads, self.swin.layers[3].blocks[0].attn.num_heads]

        for p in self.swin.parameters():
            p.requires_grad = False

    def build_peft(self, peft_mode="P5_hybrid_conv_rpb", rank=16, scale=0.2, dropout=0.0):
        self.peft_mode = peft_mode
        if peft_mode == "P0_frozen":
            return None
        return SwinPEFTModule(
            self.stage_dims, self.stage_depths, self.stage_heads,
            peft_mode=peft_mode, fusion_mode=self.fusion_mode,
            window_size=7, rank=rank, scale=scale, dropout=dropout, swin_model=self.swin
        )

    def _run_block_with_peft(self, blk, x, peft_mod, idx, mode):
        if peft_mod is None or mode == "P0_frozen":
            return blk(x)
        elif mode == "P2_adapter":
            orig_mlp_fwd = blk.mlp.forward
            down, up = peft_mod.down_list[idx], peft_mod.up_list[idx]
            s = peft_mod.scale
            blk.mlp.forward = lambda h, *a, _mlp=orig_mlp_fwd, _d=down, _u=up, _s=s, **kw: _mlp(h, *a, **kw) + _s * _u(peft_mod.drop(peft_mod.act(_d(h))))
            try: return blk(x)
            finally: blk.mlp.forward = orig_mlp_fwd
        elif mode == "P4_convpass":
            orig_mlp_fwd = blk.mlp.forward
            cp = peft_mod.convpass_list[idx]
            s = peft_mod.scale
            blk.mlp.forward = lambda h, *a, _mlp=orig_mlp_fwd, _cp=cp, _s=s, **kw: _mlp(h, *a, **kw) + _s * _cp(h)
            try: return blk(x)
            finally: blk.mlp.forward = orig_mlp_fwd
        elif mode == "P5_hybrid_conv_rpb":
            orig_mlp_fwd = blk.mlp.forward
            orig_get_rpb = blk.attn._get_rel_pos_bias
            cp = peft_mod.convpass_list[idx]
            d_rpb = peft_mod.delta_rpb[idx]
            s = peft_mod.scale
            blk.mlp.forward = lambda h, *a, _mlp=orig_mlp_fwd, _cp=cp, _s=s, **kw: _mlp(h, *a, **kw) + _s * _cp(h)
            def _custom_rpb(_orig=orig_get_rpb, _attn=blk.attn, _drpb=d_rpb):
                bias = _drpb[_attn.relative_position_index.view(-1)].view(
                    _attn.window_size[0] * _attn.window_size[1],
                    _attn.window_size[0] * _attn.window_size[1], -1
                ).permute(2, 0, 1).contiguous().unsqueeze(0)
                return _orig() + bias.to(_attn.relative_position_bias_table.dtype)
            blk.attn._get_rel_pos_bias = _custom_rpb
            try: return blk(x)
            finally:
                blk.mlp.forward = orig_mlp_fwd
                blk.attn._get_rel_pos_bias = orig_get_rpb
        return blk(x)

    def forward(self, x, peft_mod=None):
        mode = getattr(peft_mod, "peft_mode", self.peft_mode) if peft_mod is not None else "P0_frozen"
        with torch.no_grad():
            x = self.swin.patch_embed(x)
            x = self.swin.layers[0](x)
            x = self.swin.layers[1](x)

        # Stage 3: downsample maps 192 -> 384 before blocks
        layer3 = self.swin.layers[2]
        if layer3.downsample is not None:
            x = layer3.downsample(x)
        for b_idx, blk in enumerate(layer3.blocks):
            x = self._run_block_with_peft(blk, x, peft_mod, b_idx, mode)
        s3_tokens = x

        # Stage 4: downsample maps 384 -> 768 before blocks
        layer4 = self.swin.layers[3]
        if layer4.downsample is not None:
            x = layer4.downsample(x)
        off = len(layer3.blocks)
        for b_idx, blk in enumerate(layer4.blocks):
            x = self._run_block_with_peft(blk, x, peft_mod, off + b_idx, mode)
        s4_tokens = self.swin.norm(x)

        f4 = s4_tokens.flatten(1, 2).mean(dim=1)
        if self.fusion_mode == "F1_gap":
            return f4
        elif self.fusion_mode == "F2_avgmax":
            f4_max = s4_tokens.flatten(1, 2).max(dim=1)[0]
            return f4 + 0.35 * f4_max
        elif self.fusion_mode == "F3_multistage":
            f3 = F.layer_norm(s3_tokens, (self.s3_dim,)).flatten(1, 2).mean(dim=1)
            f3_rms = f3.norm(p=2, dim=-1, keepdim=True) / math.sqrt(self.s3_dim) + 1e-6
            f4_rms = f4.norm(p=2, dim=-1, keepdim=True) / math.sqrt(self.s4_dim) + 1e-6
            return torch.cat([f3 * (f4_rms / f3_rms), f4], dim=-1)

class CosineHead(nn.Module):
    def __init__(self, dim=768, num_classes=20):
        super().__init__()
        self.dim = dim
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.fc = nn.Linear(dim, num_classes, bias=False)
        self.scale = math.sqrt(768.0 / float(dim))

    def forward(self, f):
        return (self.norm(f) * self.scale) @ F.normalize(self.fc.weight, p=2, dim=1, eps=1e-6).t()

def train_first_session_peft(backbone_mod, peft_mod, task1_dataset, n_cls_t1, in_features, cfg):
    if peft_mod is None:
        return 0.0, 0.0

    peft_lr = cfg.get("peft_lr", 0.002)
    head_lr_ratio = cfg.get("head_lr_ratio", 2.0)
    head_lr = peft_lr * head_lr_ratio
    batch_size = cfg.get("batch_size", 128)
    epochs = cfg.get("epochs", 30)
    warmup_epochs = cfg.get("warmup_epochs", 2)
    wd = cfg.get("weight_decay", 1e-4)
    use_cos = cfg.get("use_cosine_sched", True)

    temp_head = CosineHead(dim=in_features, num_classes=n_cls_t1).to(DEVICE)
    if wd > 0.0:
        opt = torch.optim.AdamW([
            {"params": list(peft_mod.parameters()), "lr": peft_lr, "weight_decay": wd},
            {"params": list(temp_head.parameters()), "lr": head_lr, "weight_decay": 0.0}
        ], betas=(0.9, 0.999))
    else:
        opt = torch.optim.Adam([
            {"params": list(peft_mod.parameters()), "lr": peft_lr, "weight_decay": 0.0},
            {"params": list(temp_head.parameters()), "lr": head_lr, "weight_decay": 0.0}
        ], betas=(0.9, 0.999))

    def lr_schedule_lambda(current_ep):
        ep = current_ep + 1
        if warmup_epochs > 0 and ep <= warmup_epochs:
            return max(float(ep) / float(warmup_epochs), 0.05)
        if not use_cos:
            return 1.0
        progress = float(ep - warmup_epochs) / float(max(epochs - warmup_epochs, 1))
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_schedule_lambda)

    criterion = nn.CrossEntropyLoss()
    bs1 = min(batch_size, max(len(task1_dataset), 1))
    loader1 = DataLoader(task1_dataset, batch_size=bs1, shuffle=True, drop_last=False,
                         num_workers=0, pin_memory=True)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE == "cuda")

    p_grads, c_grads = [], []
    epoch_history = []
    for ep in range(1, epochs + 1):
        backbone_mod.eval()
        peft_mod.train()
        temp_head.train()
        tot_loss, ep_pg, ep_cg, n_b = 0.0, 0.0, 0.0, 0
        for x, y in loader1:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):
                feat = backbone_mod(x, peft_mod=peft_mod)
                logits = temp_head(feat)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            pg = peft_mod.grad_norm()
            cg = temp_head.fc.weight.grad.detach().norm(2).item() if temp_head.fc.weight.grad is not None else 0.0
            if math.isnan(pg) or math.isinf(pg):
                pg = 0.0
            if math.isnan(cg) or math.isinf(cg):
                cg = 0.0
            scaler.step(opt)
            scaler.update()

            ep_pg += pg
            ep_cg += cg
            n_b += 1
            tot_loss += loss.item()

        scheduler.step()

        avg_pg = ep_pg / max(n_b, 1)
        avg_cg = ep_cg / max(n_b, 1)
        p_grads.append(avg_pg)
        c_grads.append(avg_cg)

        cur_lr = opt.param_groups[0]["lr"]
        avg_loss = tot_loss / max(n_b, 1)
        epoch_history.append({
            "epoch": ep,
            "lr": round(float(cur_lr), 6),
            "loss": round(float(avg_loss), 4),
            "p_grad": round(float(avg_pg), 6),
            "c_grad": round(float(avg_cg), 6)
        })
        # Ghi và in gradient chính xác từng epoch một để quan sát động học
        print(f"    [Task 1 PEFT] Ep {ep:2d}/{epochs} | LR: {cur_lr:.5f} | Loss: {avg_loss:.4f} | p_grad: {avg_pg:.6f} | c_grad: {avg_cg:.6f}")

    peft_mod.save_trained_state()
    for p in peft_mod.parameters():
        p.requires_grad = False
    peft_mod.eval()
    del temp_head
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return round(float(np.mean(p_grads)), 6), round(float(np.mean(c_grads)), 6), epoch_history


In [ ]:
# ==============================================================================
# CELL 6: NCM CLASSIFIER & BỘ METRICS ĐÁNH GIÁ HÌNH HỌC TỐI ƯU SIÊU TỐC TRÊN GPU
# ==============================================================================
@torch.no_grad()
def compute_feature_eff_rank(f_mat):
    f_c = f_mat - f_mat.mean(dim=0, keepdim=True)
    cov = (f_c.t() @ f_c) / max(f_c.shape[0] - 1, 1)
    evals = torch.linalg.eigvalsh(cov.float()).clamp(min=1e-8)
    p = evals / evals.sum()
    entropy = -(p * torch.log(p)).sum().item()
    return float(torch.exp(torch.tensor(entropy)).item())

@torch.no_grad()
def compute_fisher_discriminant_ratios(features, labels, t1_classes):
    all_means = features.mean(dim=0, keepdim=True)
    n_classes = int(labels.max().item()) + 1
    dim = features.shape[1]
    dev = features.device
    dt = features.dtype

    c_sum = torch.zeros(n_classes, dim, device=dev, dtype=dt)
    c_cnt = torch.zeros(n_classes, device=dev, dtype=dt)
    ones = torch.ones(len(labels), device=dev, dtype=dt)
    c_sum.index_add_(0, labels, features)
    c_cnt.index_add_(0, labels, ones)

    valid_mask = (c_cnt > 1)
    if not valid_mask.any(): return 0.0, 0.0
    valid_classes = torch.where(valid_mask)[0]
    c_means = c_sum[valid_classes] / c_cnt[valid_classes].unsqueeze(1)
    sb_per_class = c_cnt[valid_classes] * ((c_means - all_means) ** 2).sum(dim=1)
    feat_sq = (features ** 2).sum(dim=1)
    sq_sum = torch.zeros(n_classes, device=dev, dtype=dt)
    sq_sum.index_add_(0, labels, feat_sq)
    sw_per_class = (sq_sum[valid_classes] - c_cnt[valid_classes] * (c_means ** 2).sum(dim=1)).clamp(min=1e-8)
    t1_mask = (valid_classes < t1_classes)
    unseen_mask = (valid_classes >= t1_classes)
    fdr_t1 = (sb_per_class[t1_mask].sum() / sw_per_class[t1_mask].sum().clamp(min=1e-6)).item() if t1_mask.any() else 0.0
    fdr_unseen = (sb_per_class[unseen_mask].sum() / sw_per_class[unseen_mask].sum().clamp(min=1e-6)).item() if unseen_mask.any() else 0.0
    return round(float(fdr_t1), 4), round(float(fdr_unseen), 4)

@torch.no_grad()
def compute_class_prototypes(features, labels):
    n_classes = int(labels.max().item()) + 1
    dim = features.shape[1]
    dev = features.device
    dt = features.dtype

    proto_sum = torch.zeros(n_classes, dim, device=dev, dtype=dt)
    proto_cnt = torch.zeros(n_classes, device=dev, dtype=dt)
    ones = torch.ones(len(labels), device=dev, dtype=dt)
    proto_sum.index_add_(0, labels, features)
    proto_cnt.index_add_(0, labels, ones)

    valid_mask = (proto_cnt > 0)
    valid_classes = torch.where(valid_mask)[0]
    proto_mean = proto_sum[valid_classes] / proto_cnt[valid_classes].unsqueeze(1)
    proto_norm = F.normalize(proto_mean, p=2, dim=1)
    return proto_norm, valid_classes

@torch.no_grad()
def evaluate_ncm(proto_info, test_features, test_labels, t1_classes=20):
    proto_norm, valid_classes = proto_info
    f_norm = F.normalize(test_features, p=2, dim=1)
    cos_sim = f_norm @ proto_norm.t()
    pred_idx = cos_sim.argmax(dim=1)
    preds = valid_classes[pred_idx]

    correct = (preds == test_labels).float()
    acc_all = correct.mean().item() * 100.0
    mask_t1 = (test_labels < t1_classes)
    acc_t1 = correct[mask_t1].mean().item() * 100.0 if mask_t1.any() else 0.0
    mask_unseen = (test_labels >= t1_classes)
    acc_unseen = correct[mask_unseen].mean().item() * 100.0 if mask_unseen.any() else 0.0
    return round(acc_t1, 2), round(acc_unseen, 2), round(acc_all, 2)


In [ ]:
# ==============================================================================
# CELL 7: ENGINE TRÍCH XUẤT ĐẶC TRƯNG NHANH VỚI PYTORCH AMP
# ==============================================================================
@torch.no_grad()
def extract_features_loader(backbone, peft_mod, data_loader):
    backbone.eval()
    if peft_mod is not None:
        peft_mod.eval()
    feats, targets = [], []
    for x, y in data_loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):
            f = backbone(x, peft_mod=peft_mod)
        feats.append(f.detach())
        targets.append(y.to(DEVICE, non_blocking=True))
    return torch.cat(feats, dim=0).float(), torch.cat(targets, dim=0)


In [ ]:
# ==============================================================================
# CELL 8: THỰC THI QUÉT TỪNG SIÊU THAM SỐ (11-AXIS UNIVARIATE) & KIỂM CHỨNG TỔ HỢP KẾT HỢP
# ==============================================================================
def format_axis_table(axis_name, param_name, results_list, is_multiseed=False):
    if is_multiseed:
        headers = [f"Value ({param_name})", "NCM_All (Mean±Std)", "NCM_T1", "NCM_Unseen", "FDR_Unseen", "EffRank", "p_grad"]
        widths = [20, 20, 18, 18, 12, 10, 12]
        row_fmt = " | ".join([f"{{:<{w}}}" for w in widths])
        sep = "-+-".join(["-" * w for w in widths])
        lines = [f"\n📊 BẢNG KHẢO SÁT ĐỘ NHẠY TRỤC (3 SEEDS): {axis_name.upper()} (Biến: {param_name})", row_fmt.format(*headers), sep]
        for r in results_list:
            all_str = f"{r['ncm_all']:.2f} ± {r.get('std_all', 0.0):.2f}%"
            t1_str = f"{r['ncm_t1']:.2f} ± {r.get('std_t1', 0.0):.2f}%"
            uns_str = f"{r['ncm_unseen']:.2f} ± {r.get('std_unseen', 0.0):.2f}%"
            lines.append(row_fmt.format(
                str(r["val"])[:widths[0]], all_str, t1_str, uns_str,
                f"{r['fdr_unseen']:.4f}", f"{r['eff_rank']:.1f}", f"{r['p_grad']:.6f}"
            ))
    else:
        headers = [f"Value ({param_name})", "NCM_All", "NCM_T1", "NCM_Unseen", "FDR_Unseen", "EffRank", "p_grad"]
        widths = [20, 10, 10, 12, 12, 10, 12]
        row_fmt = " | ".join([f"{{:<{w}}}" for w in widths])
        sep = "-+-".join(["-" * w for w in widths])
        lines = [f"\n📊 BẢNG KHẢO SÁT ĐỘ NHẠY TRỤC: {axis_name.upper()} (Biến: {param_name})", row_fmt.format(*headers), sep]
        for r in results_list:
            lines.append(row_fmt.format(
                str(r["val"])[:widths[0]], f"{r['ncm_all']:.2f}%", f"{r['ncm_t1']:.2f}%",
                f"{r['ncm_unseen']:.2f}%", f"{r['fdr_unseen']:.4f}", f"{r['eff_rank']:.1f}", f"{r['p_grad']:.6f}"
            ))
    return "\n".join(lines)

# ==============================================================================
# CẤU HÌNH KIỂM CHỨNG TƯƠNG TÁC CHÉO (2D JOINT GRID SEARCH)
# ==============================================================================
JOINT_CANDIDATES = OrderedDict([
    ("J0_Anchor_BS128", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=128, peft_lr=0.005, head_lr_ratio=1.0, epochs=50, warmup_epochs=0,
        scale=0.5, rank=16, adapter_dropout=0.0, weight_decay=0.0, use_cosine_sched=False, ema_coeff=0.70,
        desc="[Mốc Exp5 cũ] BS=128, lr=5e-3, r=16, s=0.5, ep=50, ema=0.7 (72.44%)"
    )),
    ("J1_BS32_SafeTuned", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=32, peft_lr=0.002, head_lr_ratio=2.0, epochs=35, warmup_epochs=2,
        scale=0.5, rank=32, adapter_dropout=0.1, weight_decay=1e-4, use_cosine_sched=True, ema_coeff=0.85,
        desc="[Ứng viên 1] BS=32, lr=2e-3, ratio=2, r=32, s=0.5, ep=35, warm=2, ema=0.85"
    )),
    ("J2_BS32_HighRank64", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=32, peft_lr=0.003, head_lr_ratio=2.5, epochs=40, warmup_epochs=2,
        scale=0.8, rank=64, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=True, ema_coeff=0.90,
        desc="[Ứng viên 2] BS=32, lr=3e-3, ratio=2.5, r=64, s=0.8, wd=0.01, drop=0.1, ema=0.9"
    )),
    ("J3_BS64_Balanced", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=64, peft_lr=0.005, head_lr_ratio=2.5, epochs=40, warmup_epochs=2,
        scale=0.8, rank=64, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=True, ema_coeff=0.90,
        desc="[Ứng viên 3] BS=64, lr=5e-3, ratio=2.5, r=64, s=0.8, wd=0.01, drop=0.1, ema=0.9"
    )),
    ("J4_JSON_RawBest", dict(
        backbone="swin_small", fusion="F3_multistage", peft="P5_hybrid_conv_rpb",
        batch_size=32, peft_lr=0.020, head_lr_ratio=5.0, epochs=50, warmup_epochs=2,
        scale=1.0, rank=64, adapter_dropout=0.1, weight_decay=0.01, use_cosine_sched=False, ema_coeff=1.00,
        desc="[Raw JSON Best] BS=32, lr=2e-2, ratio=5, r=64, s=1.0, ep=50, ema=1.0"
    )),
])

# Danh sách ứng viên được kích hoạt trong Giai đoạn 2 (có thể thêm "J0_Anchor_BS128" nếu muốn đối chiếu)
ACTIVE_JOINT_EXPERIMENTS = [
    "J0_Anchor_BS128",
    "J1_BS32_SafeTuned",
    "J2_BS32_HighRank64",
    "J3_BS64_Balanced",
    "J4_JSON_RawBest",
]

def run_univariate_hyperparameter_tuning(dataset_key=ACTIVE_DATASET):
    spec = DATASET_SPECS[dataset_key]
    d_name = spec["dataset_name"]
    n_cls = spec["n_classes"]
    task_sizes = spec["task_sizes"]
    bounds = []
    curr = 0
    for sz in task_sizes:
        bounds.append((curr, curr + sz))
        curr += sz
    t1_lo, t1_hi = bounds[0]

    print("=" * 80)
    print(f"🚀 BẮT ĐẦU SWIN HYPERPARAMETER TUNING TRÊN DATASET: {d_name} (Lớp: {n_cls})")
    print(f"📌 Chế độ thực thi: {EXECUTION_MODE}")
    print(f"📌 Active Seeds   : {ACTIVE_SEEDS} ({len(ACTIVE_SEEDS)} seeds)")
    print(f"📌 Anchor Config  : {DEFAULT_ANCHOR_CONFIG}")
    print("=" * 80)

    train_tf, eval_tf = build_transforms(spec["aug_type"])
    raw_train, raw_test = load_raw_dataset_samples(dataset_key, spec)
    is_cifar = (dataset_key == "cifar100")

    total_train_samples = len(raw_train[1]) if is_cifar else len(raw_train)
    raw_targets = raw_train[1] if is_cifar else [s[1] for s in raw_train]
    all_images_tensor = None
    all_raw_targets = None
    mmap_path = None
    use_disk_mmap = (total_train_samples >= 10000)

    if not use_disk_mmap:
        # Datasets vừa & nhỏ (< 10,000 ảnh: cars196, cub200): Nạp trực tiếp vào RAM (Zero-Disk I/O)
        print(f"[*] Đang tiền tính toán toàn bộ {total_train_samples} ảnh vào RAM (Zero-Disk I/O)...")
        t0_pre = time.time()
        all_eval_tensors = []
        for idx in range(total_train_samples):
            p, _ = raw_train[idx]
            img = Image.open(p).convert("RGB")
            all_eval_tensors.append(eval_tf(img))
        all_images_tensor = torch.stack(all_eval_tensors, dim=0)
        all_raw_targets = torch.tensor(raw_targets, dtype=torch.long)
        ram_mb = (all_images_tensor.element_size() * all_images_tensor.nelement()) / (1024 * 1024)
        print(f"    -> Hoàn tất nạp {all_images_tensor.shape[0]} tensor vào RAM trong {time.time() - t0_pre:.1f}s (RAM sử dụng: {ram_mb:.1f} MB)")
    else:
        # Datasets lớn (>= 10,000 ảnh: cifar100: 50k, imagenet_r: 24k): Tự động dùng bộ đệm đĩa Uint8 trên /tmp
        # Triệt tiêu hoàn toàn nguy cơ sập OOM 30 GB RAM trên Kaggle!
        mmap_dir = "/tmp" if os.path.exists("/tmp") else tempfile.gettempdir()
        mmap_path = os.path.join(mmap_dir, f"{dataset_key}_train_resized_224.mmap")
        expected_bytes = total_train_samples * 224 * 224 * 3
        expected_gb = expected_bytes / (1024 * 1024 * 1024)
        needs_build = True

        if os.path.exists(mmap_path):
            if os.path.getsize(mmap_path) == expected_bytes:
                needs_build = False
                print(f"[*] Đã tìm thấy bộ đệm đĩa {expected_gb:.2f} GB Uint8 tại: {mmap_path} (Tái sử dụng tức thì)")
            else:
                try:
                    os.remove(mmap_path)
                except Exception:
                    pass

        if needs_build:
            print(f"[*] Đang tiền xử lý & tạo bộ đệm đĩa {expected_gb:.2f} GB Uint8 trên /tmp ({mmap_path})...")
            t0_cache = time.time()
            mmap_fp = np.memmap(mmap_path, dtype=np.uint8, mode='w+', shape=(total_train_samples, 224, 224, 3))
            resize_crop = T.Compose([
                T.Resize((256, 256)),
                T.CenterCrop(224)
            ])
            chunk_sz = 5000
            for start_i in range(0, total_train_samples, chunk_sz):
                end_i = min(start_i + chunk_sz, total_train_samples)
                for i in range(start_i, end_i):
                    if is_cifar:
                        img_pil = Image.fromarray(raw_train[0][i])
                    else:
                        img_pil = Image.open(raw_train[i][0]).convert("RGB")
                    mmap_fp[i] = np.array(resize_crop(img_pil), dtype=np.uint8)
                print(f"    -> Đã ghi {end_i}/{total_train_samples} ảnh vào đệm đĩa ({(end_i * 224 * 224 * 3) / (1024 * 1024):.1f} MB)...")
            mmap_fp.flush()
            del mmap_fp
            print(f"    -> Hoàn tất khởi tạo bộ đệm {expected_gb:.2f} GB trong {time.time() - t0_cache:.1f}s (RAM: < 1 GB, Disk: {expected_gb:.2f} GB)")

    # Bộ đệm dữ liệu theo seed (tránh tính toán lại val_loader & t1_ds qua các trục)
    seed_cache = {}
    def get_seed_data(seed):
        if seed in seed_cache:
            return seed_cache[seed]
        order, remap, tr_idx, val_idx = setup_cil_seed(raw_train, seed, n_cls, task_sizes, n_splits=5, is_cifar=is_cifar)
        t1_train_idx = [i for i in tr_idx if remap[raw_targets[i]] < t1_hi]
        t1_ds = CachedTrainSubset(raw_train, t1_train_idx, remap, train_tf, is_cifar=is_cifar)

        if use_disk_mmap:
            # Datasets lớn (cifar100: 7.5 GB, imagenet_r: 3.6 GB): Memory-mapping trực tiếp từ /tmp
            val_ds = Uint8MmapEvalDataset(mmap_path, val_idx, remap, raw_targets, shape=(total_train_samples, 224, 224, 3), aug_type=spec["aug_type"])
            val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
            all_tr_ds = Uint8MmapEvalDataset(mmap_path, tr_idx, remap, raw_targets, shape=(total_train_samples, 224, 224, 3), aug_type=spec["aug_type"])
            all_tr_loader = DataLoader(all_tr_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
        else:
            remap_t = torch.tensor([remap[y.item()] for y in all_raw_targets])
            val_ds = PreloadedTensorDataset(all_images_tensor[val_idx], remap_t[val_idx])
            val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
            all_tr_ds = PreloadedTensorDataset(all_images_tensor[tr_idx], remap_t[tr_idx])
            all_tr_loader = DataLoader(all_tr_ds, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

        seed_cache[seed] = (val_loader, all_tr_loader, t1_ds)
        return seed_cache[seed]

    full_tuning_report = {}
    best_discovered_params = dict(DEFAULT_ANCHOR_CONFIG)
    is_multi = (len(ACTIVE_SEEDS) > 1)

    # ==========================================================================
    # GIAI ĐOẠN 1: QUÉT ĐƠN BIẾN 11 TRỤC (ONE-FACTOR-AT-A-TIME)
    # ==========================================================================
    if EXECUTION_MODE in ["FULL_SWEEP_AND_JOINT", "UNIVARIATE_ONLY"]:
        axes_to_sweep = TUNING_AXES.keys() if ACTIVE_TUNING_AXES == "ALL_AXES" else ACTIVE_TUNING_AXES
        print(f"\n[*] Bắt đầu Giai đoạn 1: Quét {len(axes_to_sweep)} trục siêu tham số độc lập...")

        for axis_key in axes_to_sweep:
            if axis_key not in TUNING_AXES:
                print(f"[!] Bỏ qua trục không tồn tại: {axis_key}")
                continue

            axis_info = TUNING_AXES[axis_key]
            p_name = axis_info["param_name"]
            p_values = axis_info["values"]
            desc = axis_info["desc"]

            print(f"\n" + "=" * 70)
            print(f"🔍 [TRỤC KHẢO SÁT] {axis_key.upper()} -> {desc}")
            print(f"   Các giá trị khảo sát: {p_values}")
            print("=" * 70)

            axis_results = []

            # Trục ema_coeff: Quét siêu nhanh trong 0 giây (không cần train lại)!
            if axis_key == "ema_coeff":
                cfg = dict(DEFAULT_ANCHOR_CONFIG)
                bb_name, fs_mode, pf_mode = cfg["backbone"], cfg["fusion"], cfg["peft"]
                rank_val, scale_val, drop_val = cfg["rank"], cfg["scale"], cfg["adapter_dropout"]

                seed_runs = {}
                for seed in ACTIVE_SEEDS:
                    val_loader, all_tr_loader, t1_ds = get_seed_data(seed)

                    backbone = SwinBackboneWrapper(
                        timm_name=SWIN_ARCH_MAP[bb_name], fusion_mode=fs_mode, peft_mode=pf_mode,
                        rank=rank_val, scale=scale_val, dropout=drop_val
                    ).to(DEVICE)
                    peft_mod = backbone.build_peft(peft_mode=pf_mode, rank=rank_val, scale=scale_val, dropout=drop_val).to(DEVICE)
                    in_feat_dim = backbone.stage_dims[0] + backbone.stage_dims[1] if fs_mode == "F3_multistage" else backbone.stage_dims[1]

                    p_grad, c_grad, ep_history = train_first_session_peft(backbone, peft_mod, t1_ds, t1_hi - t1_lo, in_feat_dim, cfg)
                    seed_runs[seed] = (backbone, peft_mod, all_tr_loader, val_loader, p_grad, c_grad, ep_history)

                for val in p_values:
                    val_metrics = []
                    for seed, (bb, pm, tr_ld, val_ld, pg, cg, eh) in seed_runs.items():
                        pm.apply_ema_anchoring(val)
                        tr_f, tr_y = extract_features_loader(bb, pm, tr_ld)
                        val_f, val_y = extract_features_loader(bb, pm, val_ld)
                        protos = compute_class_prototypes(tr_f, tr_y)
                        a_t1, a_uns, a_all = evaluate_ncm(protos, val_f, val_y, t1_classes=t1_hi)
                        f_t1, f_uns = compute_fisher_discriminant_ratios(val_f, val_y, t1_hi)
                        e_rk = compute_feature_eff_rank(val_f)
                        val_metrics.append({"acc_all": a_all, "acc_t1": a_t1, "acc_unseen": a_uns, "fdr_unseen": f_uns, "eff_rank": e_rk, "p_grad": pg, "c_grad": cg})

                    m_all, s_all = float(np.mean([m["acc_all"] for m in val_metrics])), float(np.std([m["acc_all"] for m in val_metrics]))
                    m_t1, s_t1 = float(np.mean([m["acc_t1"] for m in val_metrics])), float(np.std([m["acc_t1"] for m in val_metrics]))
                    m_uns, s_uns = float(np.mean([m["acc_unseen"] for m in val_metrics])), float(np.std([m["acc_unseen"] for m in val_metrics]))
                    m_fdr = float(np.mean([m["fdr_unseen"] for m in val_metrics]))
                    m_rk = float(np.mean([m["eff_rank"] for m in val_metrics]))
                    m_pg = float(np.mean([m["p_grad"] for m in val_metrics]))
                    m_cg = float(np.mean([m["c_grad"] for m in val_metrics]))

                    record = {
                        "val": val, "ncm_all": m_all, "std_all": s_all, "ncm_t1": m_t1, "std_t1": s_t1,
                        "ncm_unseen": m_uns, "std_unseen": s_uns, "fdr_unseen": m_fdr, "eff_rank": m_rk,
                        "p_grad": m_pg, "c_grad": m_cg, "seed_metrics": val_metrics
                    }
                    axis_results.append(record)
                    std_str = f" ± {s_all:.2f}" if is_multi else ""
                    print(f"  [{p_name} = {val}] NCM All: {m_all:.2f}%{std_str} | Unseen: {m_uns:.2f}% | FDR_Unseen: {m_fdr:.4f} | EffRank: {m_rk:.1f}")

                for seed, (bb, pm, *_) in seed_runs.items():
                    del bb, pm
                if DEVICE == "cuda": torch.cuda.empty_cache()

            else:
                for val in p_values:
                    val_metrics = []
                    cfg = dict(DEFAULT_ANCHOR_CONFIG)
                    cfg[p_name] = val
                    print(f"\n  >>> Thử nghiệm {p_name} = {val} trên {len(ACTIVE_SEEDS)} seed(s): {ACTIVE_SEEDS} ...")

                    for seed in ACTIVE_SEEDS:
                        val_loader, all_tr_loader, t1_ds = get_seed_data(seed)

                        bb_name, fs_mode, pf_mode = cfg["backbone"], cfg["fusion"], cfg["peft"]
                        rank_val, scale_val, drop_val = cfg["rank"], cfg["scale"], cfg["adapter_dropout"]

                        backbone = SwinBackboneWrapper(
                            timm_name=SWIN_ARCH_MAP[bb_name], fusion_mode=fs_mode, peft_mode=pf_mode,
                            rank=rank_val, scale=scale_val, dropout=drop_val
                        ).to(DEVICE)
                        peft_mod = backbone.build_peft(peft_mode=pf_mode, rank=rank_val, scale=scale_val, dropout=drop_val).to(DEVICE)
                        in_feat_dim = backbone.stage_dims[0] + backbone.stage_dims[1] if fs_mode == "F3_multistage" else backbone.stage_dims[1]

                        if is_multi:
                            print(f"    --- [Seed {seed}] (Val Fold {seed}) ---")

                        p_grad, c_grad, ep_history = train_first_session_peft(backbone, peft_mod, t1_ds, t1_hi - t1_lo, in_feat_dim, cfg)
                        peft_mod.apply_ema_anchoring(cfg["ema_coeff"])

                        t_ev0 = time.time()
                        tr_feats, tr_labels = extract_features_loader(backbone, peft_mod, all_tr_loader)
                        val_feats, val_labels = extract_features_loader(backbone, peft_mod, val_loader)

                        prototypes = compute_class_prototypes(tr_feats, tr_labels)
                        acc_t1, acc_unseen, acc_all = evaluate_ncm(prototypes, val_feats, val_labels, t1_classes=t1_hi)
                        fdr_t1, fdr_unseen = compute_fisher_discriminant_ratios(val_feats, val_labels, t1_hi)
                        eff_rk = compute_feature_eff_rank(val_feats)
                        t_eval_sec = time.time() - t_ev0

                        val_metrics.append({
                            "seed": seed, "acc_all": acc_all, "acc_t1": acc_t1, "acc_unseen": acc_unseen,
                            "fdr_unseen": fdr_unseen, "eff_rank": eff_rk, "p_grad": p_grad, "c_grad": c_grad,
                            "epoch_history": ep_history
                        })

                        del backbone, peft_mod
                        if DEVICE == "cuda": torch.cuda.empty_cache()

                    m_all, s_all = float(np.mean([m["acc_all"] for m in val_metrics])), float(np.std([m["acc_all"] for m in val_metrics]))
                    m_t1, s_t1 = float(np.mean([m["acc_t1"] for m in val_metrics])), float(np.std([m["acc_t1"] for m in val_metrics]))
                    m_uns, s_uns = float(np.mean([m["acc_unseen"] for m in val_metrics])), float(np.std([m["acc_unseen"] for m in val_metrics]))
                    m_fdr = float(np.mean([m["fdr_unseen"] for m in val_metrics]))
                    m_rk = float(np.mean([m["eff_rank"] for m in val_metrics]))
                    m_pg = float(np.mean([m["p_grad"] for m in val_metrics]))
                    m_cg = float(np.mean([m["c_grad"] for m in val_metrics]))

                    record = {
                        "val": val, "ncm_all": m_all, "std_all": s_all, "ncm_t1": m_t1, "std_t1": s_t1,
                        "ncm_unseen": m_uns, "std_unseen": s_uns, "fdr_unseen": m_fdr, "eff_rank": m_rk,
                        "p_grad": m_pg, "c_grad": m_cg, "seed_metrics": val_metrics
                    }
                    axis_results.append(record)
                    std_str = f" ± {s_all:.2f}" if is_multi else ""
                    print(f"  => [{p_name} = {val}] NCM All: {m_all:.2f}%{std_str} | Unseen: {m_uns:.2f}% | FDR_Unseen: {m_fdr:.4f} | EffRank: {m_rk:.1f} | p_grad: {m_pg:.6f} (Đánh giá GPU: {t_eval_sec:.1f}s)")

            print(format_axis_table(axis_key, p_name, axis_results, is_multiseed=is_multi))

            best_r = max(axis_results, key=lambda x: (x["ncm_all"], x["fdr_unseen"]))
            print(f"  🏆 Giá trị tối ưu cho {p_name}: {best_r['val']} (NCM All: {best_r['ncm_all']:.2f}%, FDR Unseen: {best_r['fdr_unseen']:.4f})")
            best_discovered_params[p_name] = best_r["val"]
            full_tuning_report[axis_key] = {
                "param_name": p_name,
                "best_value": best_r["val"],
                "sweep_results": axis_results
            }

        # Báo cáo tổng hợp bộ siêu tham số tốt nhất từng trục
        print("\n" + "=" * 80)
        print("🎯 BỘ SIÊU THAM SỐ TỐT NHẤT TỪNG TRỤC ĐỘC LẬP (UNIVARIATE BEST):")
        print("=" * 80)
        for k, v in best_discovered_params.items():
            print(f"  * {k:<18}: {v}")

    # ==========================================================================
    # GIAI ĐOẠN 2: KIỂM CHỨNG TỔ HỢP KẾT HỢP (JOINT COMBINATION VERIFICATION)
    # ==========================================================================
    joint_results = []
    champion = None

    if EXECUTION_MODE in ["FULL_SWEEP_AND_JOINT", "JOINT_ONLY"]:
        candidates_to_run = [k for k in ACTIVE_JOINT_EXPERIMENTS if k in JOINT_CANDIDATES]
        print("\n" + "=" * 80)
        print("🔬 BẮT ĐẦU GIAI ĐOẠN 2: KIỂM CHỨNG TỔ HỢP KẾT HỢP (JOINT COMBINATION VERIFICATION)")
        print(f"📌 Đánh giá {len(candidates_to_run)} tổ hợp kết hợp trên {len(ACTIVE_SEEDS)} seed(s): {ACTIVE_SEEDS}")
        print("=" * 80)

        for cand_name in candidates_to_run:
            cfg = JOINT_CANDIDATES[cand_name]
            desc = cfg.get("desc", cand_name)
            print(f"\n>>> [Tổ hợp] {cand_name} -> {desc}")
            print(f"    Chi tiết: BS={cfg['batch_size']}, LR={cfg['peft_lr']}, HeadRatio={cfg['head_lr_ratio']}, Epochs={cfg['epochs']}, Scale={cfg['scale']}, Rank={cfg['rank']}, Sched={cfg['use_cosine_sched']}, EMA={cfg['ema_coeff']}")
            cand_metrics = []

            for seed in ACTIVE_SEEDS:
                val_loader, all_tr_loader, t1_ds = get_seed_data(seed)

                bb_name, fs_mode, pf_mode = cfg["backbone"], cfg["fusion"], cfg["peft"]
                rank_val, scale_val, drop_val = cfg["rank"], cfg["scale"], cfg["adapter_dropout"]

                backbone = SwinBackboneWrapper(
                    timm_name=SWIN_ARCH_MAP[bb_name], fusion_mode=fs_mode, peft_mode=pf_mode,
                    rank=rank_val, scale=scale_val, dropout=drop_val
                ).to(DEVICE)
                peft_mod = backbone.build_peft(peft_mode=pf_mode, rank=rank_val, scale=scale_val, dropout=drop_val).to(DEVICE)
                in_feat_dim = backbone.stage_dims[0] + backbone.stage_dims[1] if fs_mode == "F3_multistage" else backbone.stage_dims[1]

                if is_multi:
                    print(f"    --- [Seed {seed}] (Val Fold {seed}) ---")

                p_grad, c_grad, ep_history = train_first_session_peft(backbone, peft_mod, t1_ds, t1_hi - t1_lo, in_feat_dim, cfg)
                peft_mod.apply_ema_anchoring(cfg["ema_coeff"])

                t_ev0 = time.time()
                tr_feats, tr_labels = extract_features_loader(backbone, peft_mod, all_tr_loader)
                val_feats, val_labels = extract_features_loader(backbone, peft_mod, val_loader)

                prototypes = compute_class_prototypes(tr_feats, tr_labels)
                acc_t1, acc_unseen, acc_all = evaluate_ncm(prototypes, val_feats, val_labels, t1_classes=t1_hi)
                fdr_t1, fdr_unseen = compute_fisher_discriminant_ratios(val_feats, val_labels, t1_hi)
                eff_rk = compute_feature_eff_rank(val_feats)
                t_eval_sec = time.time() - t_ev0

                cand_metrics.append({
                    "seed": seed, "acc_all": acc_all, "acc_t1": acc_t1, "acc_unseen": acc_unseen,
                    "fdr_unseen": fdr_unseen, "eff_rank": eff_rk, "p_grad": p_grad, "c_grad": c_grad,
                    "epoch_history": ep_history
                })

                del backbone, peft_mod
                if DEVICE == "cuda": torch.cuda.empty_cache()

            m_all, s_all = float(np.mean([m["acc_all"] for m in cand_metrics])), float(np.std([m["acc_all"] for m in cand_metrics]))
            m_t1, s_t1 = float(np.mean([m["acc_t1"] for m in cand_metrics])), float(np.std([m["acc_t1"] for m in cand_metrics]))
            m_uns, s_uns = float(np.mean([m["acc_unseen"] for m in cand_metrics])), float(np.std([m["acc_unseen"] for m in cand_metrics]))
            m_fdr = float(np.mean([m["fdr_unseen"] for m in cand_metrics]))
            m_rk = float(np.mean([m["eff_rank"] for m in cand_metrics]))
            m_pg = float(np.mean([m["p_grad"] for m in cand_metrics]))
            m_cg = float(np.mean([m["c_grad"] for m in cand_metrics]))

            record = {
                "candidate": cand_name, "config": cfg, "ncm_all": m_all, "std_all": s_all,
                "ncm_t1": m_t1, "std_t1": s_t1, "ncm_unseen": m_uns, "std_unseen": s_uns,
                "fdr_unseen": m_fdr, "eff_rank": m_rk, "p_grad": m_pg, "c_grad": m_cg,
                "seed_metrics": cand_metrics
            }
            joint_results.append(record)
            std_str = f" ± {s_all:.2f}" if is_multi else ""
            print(f"  => [{cand_name}] NCM All: {m_all:.2f}%{std_str} | Unseen: {m_uns:.2f}% | FDR_Unseen: {m_fdr:.4f} | EffRank: {m_rk:.1f} (Đánh giá GPU: {t_eval_sec:.1f}s)")

        # Format Bảng tổng hợp Joint Candidates
        if is_multi:
            headers = ["Candidate", "NCM_All (Mean±Std)", "NCM_T1", "NCM_Unseen", "FDR_Unseen", "EffRank", "p_grad"]
            widths = [22, 20, 18, 18, 12, 10, 12]
        else:
            headers = ["Candidate", "NCM_All", "NCM_T1", "NCM_Unseen", "FDR_Unseen", "EffRank", "p_grad"]
            widths = [22, 10, 10, 12, 12, 10, 12]

        row_fmt = " | ".join([f"{{:<{w}}}" for w in widths])
        sep = "-+-".join(["-" * w for w in widths])
        print("\n" + "=" * 90)
        print("📊 BẢNG SO SÁNH TỔ HỢP KẾT HỢP (JOINT COMBINATION BENCHMARK):")
        print(row_fmt.format(*headers))
        print(sep)
        for r in joint_results:
            if is_multi:
                all_str = f"{r['ncm_all']:.2f} ± {r['std_all']:.2f}%"
                t1_str = f"{r['ncm_t1']:.2f} ± {r['std_t1']:.2f}%"
                uns_str = f"{r['ncm_unseen']:.2f} ± {r['std_unseen']:.2f}%"
                print(row_fmt.format(r["candidate"][:widths[0]], all_str, t1_str, uns_str, f"{r['fdr_unseen']:.4f}", f"{r['eff_rank']:.1f}", f"{r['p_grad']:.6f}"))
            else:
                print(row_fmt.format(r["candidate"][:widths[0]], f"{r['ncm_all']:.2f}%", f"{r['ncm_t1']:.2f}%", f"{r['ncm_unseen']:.2f}%", f"{r['fdr_unseen']:.4f}", f"{r['eff_rank']:.1f}", f"{r['p_grad']:.6f}"))

        champion = max(joint_results, key=lambda x: (x["ncm_all"], x["fdr_unseen"]))
        print("\n" + "=" * 90)
        print(f"🏆 TỔ HỢP SIÊU THAM SỐ VÀNG CHUNG CUỘC: {champion['candidate']}")
        print(f"   NCM All: {champion['ncm_all']:.2f}% | Unseen: {champion['ncm_unseen']:.2f}% | FDR_Unseen: {champion['fdr_unseen']:.4f} | EffRank: {champion['eff_rank']:.1f}")
        print("   Cấu hình chi tiết:")
        for k, v in champion["config"].items():
            print(f"     * {k:<18}: {v}")
        print("=" * 90)

    # Xuất file kết quả JSON toàn diện
    out_json = os.path.join(MASTER_OUT_DIR, f"Swin_11Axis_Tuning_{d_name}.json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump({
            "dataset": d_name,
            "anchor_config": DEFAULT_ANCHOR_CONFIG,
            "execution_mode": EXECUTION_MODE,
            "univariate_best_params": best_discovered_params,
            "univariate_axes_report": full_tuning_report,
            "joint_candidates_report": joint_results,
            "champion_joint_config": champion["config"] if champion else best_discovered_params
        }, f, indent=2, ensure_ascii=False)
    print(f"\n[*] Đã lưu báo cáo chi tiết tại: {out_json}")

    zip_out = "/kaggle/working/Swin_Univariate_Tuning_Outputs.zip" if IS_KAGGLE else "./results/apt/Swin_Univariate_Tuning_Outputs.zip"
    if os.path.exists(MASTER_OUT_DIR):
        shutil.make_archive(zip_out.replace(".zip", ""), "zip", MASTER_OUT_DIR)
        print(f"📦 ĐÃ ĐÓNG GÓI ZIP KẾT QUẢ TẠI: {zip_out} (Sẵn sàng tải về chỉ với 1 click!)")

if __name__ == "__main__":
    run_univariate_hyperparameter_tuning(ACTIVE_DATASET)
